# Calculate total mass mixing ratios for each variable

For mass mixing ratios of small particles in CESM2, the variables are usually separated by mode - for example, the so4 variables are called so4_a1, so4_a2, so4_a3, so4_c1, so4_c2, and so4_c3, where 1-3 are accumulation, aitken, and coarse modes, respectively, and "a" variables are for dry particles and "c" are for in-cloud. Likewise, for BC, you can look for bc_a1, bc_a2, etc.

Read more [here](https://wiki.ucar.edu/spaces/camchem/pages/358319530/Aerosols)

This script calculates the total sum for each variable.

In [ ]:
import os
import re
import warnings
import xarray as xr
from collections import defaultdict
from utils.utils import get_scenario_config, load_file_list, minus_one_month

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"
SCRATCH = f"/glade/derecho/scratch/awells/air_quality/{model}/pm25/"
T_DIR = f"/glade/work/awells/air_quality/{model}/temp/temp_pres/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/monthly_pm25/"

VAR_list = ["BC", "POA", "SOA", "SO4", "SS", "DU"]

In [ ]:
def load_file(f):
    if not os.path.exists(f):
        raise ValueError(f"Missing: {f}")

    # Find variable
    pattern = re.compile(r"cam\.h0\.([^.]+)\.")
    m = pattern.search(f)
    varname = m.group(1)

    ds = xr.open_dataset(f)
    da = ds[varname]
    return da


def select_surface(da):
    if "lev" in da.dims:
        da = da.isel(lev=-1)
    else:
        print("No lev dimension")
    return da

In [ ]:
for var in VAR_list:
    for ens_num in ensemble_members:
        print(f"Processing {var}, {scenario} ensemble {ens_num:02d}")
        # Load all file lists
        file_list = load_file_list(FILE_DIR, f"file_list_{var}_{scenario}_{ens_num:02d}.json")

        groups = defaultdict(list)
        for f in file_list:
            # Load surface variable
            da = select_surface(load_file(f))
            groups[da.name].append(da)

        combined = {}
        for vbl, das in groups.items():
            combined[vbl] = xr.concat(
                das,
                dim="time",
                join="outer",
                combine_attrs="drop_conflicts"
            )

        # Get unit attribute from combined
        first_key = next(iter(combined))
        units = combined[first_key].attrs["units"]

        # Calculate the sum of all modes etc.
        total_var = sum(combined.values())
        total_var.attrs["units"] = units

        # If the first month is February (2) then apply month fixer
        first_month = total_var.time.dt.month[0]
        if first_month == 2:
            print("Adjusting month indexing")
            new_time = [minus_one_month(t) for t in total_var["time"].values]
            total_var = total_var.assign_coords(time=new_time)
        # If the first month is January (1) don't apply month fixer
        elif first_month == 1:
            print("No month index adjusting needed")
        else:
            warnings.warn(f"First month: {first_month}, check dates in file")

        first_year = total_var.time.dt.year[0].item()
        last_year = total_var.time.dt.year[-1].item()

        out_file = f"{var}_mmr_{model}_{scenario}_{ens_num:02d}_{first_year}-{last_year}.nc"
        out_path = os.path.join(SCRATCH, out_file)
        total_var.to_netcdf(out_path)

print("All processing complete.")